# Day 16 · 数据摄取

> **前置**: Day11-15 已掌握 NumPy/Pandas/可视化  
> **关键问题**: 数据不只是 CSV,还有 JSON、Excel、数据库、API 接口 —— 本节学习"从任何源头把数据读进 Pandas"。

**引入:一行代码连接任何数据源**

抽问:`plt.hist(df["年龄"])` 画什么图?(直方图)如果数据不在 DataFrame 里,而在一个 JSON 文件里,要先做什么?(读入 
DataFrame)上一节学会了"看"数据,这一节学会"取"数据 —— 从 CSV/JSON/Excel/数据库/API 任何源头。


**1. read_csv 参数详解**

**read_csv** 是最常用的读入函数,核心参数: **encoding** (文件编码,常用 utf-8 或 gbk)、**sep** (分隔符,默认逗号,制表符用 
`\t`)、**header** (第几行做列名,默认 0)、**index_col** (指定某列做索引)、**usecols** (只读部分列,节省内存)、**nrows** (只读前 
N 行)、**na_values** (哪些值视为缺失值)。

因本课不依赖外部文件,用 **io.StringIO** 把字符串当成文件来读,完全等价于 `pd.read_csv("data.csv")`。


In [ ]:
import pandas as pd
import io

# 用 io.StringIO 模拟 CSV 文件(避免依赖外部文件)
csv_data = (
    "姓名,年龄,城市\n"
    "张三,20,北京\n"
    "李四,21,上海\n"
    "王五,19,广州\n"
    "赵六,22,深圳\n"
)
df = pd.read_csv(io.StringIO(csv_data))
print("--- 基础读取 ---")
print(df)

# sep 参数:以分号做分隔符
csv_sep = "姓名;年龄;城市\n张三;20;北京\n"
df_sep = pd.read_csv(io.StringIO(csv_sep), sep=";")
print("\n--- sep=\; ---")
print(df_sep)

# index_col 参数:把"姓名"列设为行索引
df_idx = pd.read_csv(
    io.StringIO(csv_data), index_col="姓名"
)
print("\n--- index_col ---")
print(df_idx)


接下来继续演示 **usecols**、**nrows**、**na_values** 三个实用参数,分别做列筛选、行数限制、缺失值替换。


In [ ]:
import pandas as pd
import io

# 基础 CSV 数据
csv_data = (
    "姓名,年龄,城市\n"
    "张三,20,北京\n"
    "李四,21,上海\n"
    "王五,19,广州\n"
    "赵六,22,深圳\n"
)

# usecols 参数:只取姓名和年龄两列
df_cols = pd.read_csv(
    io.StringIO(csv_data), usecols=["姓名", "年龄"]
)
print("--- usecols ---")
print(df_cols)

# nrows 参数:只读前 2 行
df_n = pd.read_csv(io.StringIO(csv_data), nrows=2)
print("\n--- nrows=2 ---")
print(df_n)

# na_values 参数:把"NA"和"-"视为缺失值 NaN
csv_na = "姓名,年龄\n张三,20\n李四,NA\n王五,-\n"
df_na = pd.read_csv(
    io.StringIO(csv_na), na_values=["NA", "-"]
)
print("\n--- na_values ---")
print(df_na)


**2. read_json 与 read_excel**

**read_json** 支持 JSON 字符串或 JSON 文件路径,常见 **orient** 取值: 
"records"(列表套字典)、"index"(字典套字典)。**read_excel** 读取 .xlsx 文件,需安装 **openpyxl** 库,常用参数 
**sheet_name** (指定工作表名或序号)。本课把 read_excel 写成注释,实际运行时取消注释即可。


In [ ]:
import pandas as pd
import io

# JSON 字符串(orient="records" 格式:列表套字典)
json_str = (
    '[{"name":"张三","age":20},'
    '{"name":"李四","age":21},'
    '{"name":"王五","age":19}]'
)
df_j = pd.read_json(json_str)
print("--- 读入 JSON ---")
print(df_j)

# DataFrame 导出为 JSON 字符串
out = df_j.to_json(orient="records", force_ascii=False)
print("\n--- 导出 JSON (orient=records) ---")
print(out)

# 用 orient="index" 导出
out_idx = df_j.to_json(orient="index", force_ascii=False)
print("\n--- 导出 JSON (orient=index) ---")
print(out_idx)

# read_excel 示例(需安装 openpyxl,此处注释)
# df_excel = pd.read_excel(
#     "data.xlsx", sheet_name="Sheet1"
# )
# print(df_excel)

# 用 StringIO 模拟也能演示 read_json 读文件对象
df_io = pd.read_json(io.StringIO(json_str))
print("\n--- 通过 StringIO 读 JSON ---")
print(df_io)


**3. read_sql 读取数据库**

**read_sql** 配合 SQLite 数据库使用:先建立连接,再创建表并插入数据,最后用 SELECT 语句把查询结果读成 DataFrame。推荐用 **with** 
上下文管理器自动关闭连接,避免忘记 conn.close() 造成资源泄漏。


In [ ]:
import pandas as pd
import sqlite3

# 创建内存数据库(程序结束自动销毁)
conn = sqlite3.connect(":memory:")
cur = conn.cursor()

# 创建表
cur.execute(
    "CREATE TABLE students "
    "(id INTEGER PRIMARY KEY, name TEXT, age INTEGER)"
)

# 批量插入数据
rows = [(1, "张三", 20), (2, "李四", 21),
        (3, "王五", 19), (4, "赵六", 22)]
cur.executemany(
    "INSERT INTO students VALUES (?, ?, ?)", rows
)
conn.commit()

# 条件查询 age>20,读成 DataFrame
df_sql = pd.read_sql(
    "SELECT * FROM students WHERE age > 20", conn
)
print("--- 条件查询 age>20 ---")
print(df_sql)

# 关闭连接
conn.close()


上面手动调用了 `conn.close()`。实际项目中推荐 **with** 上下文,代码更简洁、不会遗漏关闭。下面是等效写法。


In [ ]:
import pandas as pd
import sqlite3

# 用 with 上下文管理器(更安全,自动关闭)
with sqlite3.connect(":memory:") as conn2:
    cur2 = conn2.cursor()
    cur2.execute(
        "CREATE TABLE books "
        "(title TEXT, price REAL)"
    )
    cur2.executemany(
        "INSERT INTO books VALUES (?, ?)",
        [("Python基础", 59.8),
         ("数据分析", 88.0)],
    )
    conn2.commit()
    df_books = pd.read_sql("SELECT * FROM books", conn2)
    print("--- with 上下文读取 books ---")
    print(df_books)
# with 块结束,conn2 自动关闭,无需手动 close


**4. API 数据拉取 (requests)**

真实项目中常通过 **HTTP API** 获取数据:用 **requests.get** 发请求,检查 **status_code** 是否 200(成功),再调用 
**response.json()** 拿到字典,最后转为 DataFrame。本课把请求代码全部写成注释,并说明实际用法。


In [ ]:
import pandas as pd
import json

# 真实代码示例(已注释,说明典型流程):
#
# import requests
#
# # 1. 发送 GET 请求到 API 接口
# url = "https://api.example.com/students"
# response = requests.get(url, timeout=10)
#
# # 2. 检查响应状态码(200 表示成功)
# if response.status_code == 200:
#     # 3. 解析 JSON 响应体为 Python 字典
#     data = response.json()
#     # 4. 转为 DataFrame 进行分析
#     df_api = pd.DataFrame(data)
#     print(df_api)
# else:
#     print("请求失败,状态码:", response.status_code)


# 下面用模拟数据演示"字典→DataFrame"这一步
mock_api_data = [
    {"name": "张三", "age": 20},
    {"name": "李四", "age": 21},
]
df_mock = pd.DataFrame(mock_api_data)
print("--- 模拟 API 返回的 DataFrame ---")
print(df_mock)


**5. 数据导出 (to_csv / to_json / to_sql)**

分析完成后常需把结果持久化: **to_csv** 写 CSV 文件(**index=False** 避免多出一列行号)、**to_json** 写 JSON 文件(配合 
**force_ascii=False** 保留中文)、**to_sql** 写回数据库(**if_exists** 控制表已存在时是替换 "replace" 还是追加 "append")。


In [ ]:
import pandas as pd
import sqlite3

# 准备一个示例 DataFrame
df_out = pd.DataFrame(
    {"name": ["张三", "李四"],
     "age": [20, 21]}
)

# to_csv:导出 CSV,index=False 不保留行索引
df_out.to_csv("output.csv", index=False,
              encoding="utf-8")
print("--- 已导出 output.csv ---")

# to_json:导出 JSON,orient=records 最通用
df_out.to_json("output.json", orient="records",
               force_ascii=False, indent=2)
print("--- 已导出 output.json ---")

# to_sql:把 DataFrame 写入 SQLite,再读回验证
with sqlite3.connect(":memory:") as conn:
    # if_exists="replace":表已存在则覆盖
    df_out.to_sql("people", conn,
                  if_exists="replace", index=False)
    df_check = pd.read_sql("SELECT * FROM people", conn)
    print("\n--- to_sql replace 后读取 ---")
    print(df_check)

    # if_exists="append":追加到已有表
    df_more = pd.DataFrame(
        {"name": ["王五"], "age": [19]}
    )
    df_more.to_sql("people", conn,
                   if_exists="append", index=False)
    df_all = pd.read_sql("SELECT * FROM people", conn)
    print("\n--- to_sql append 后读取 ---")
    print(df_all)


**今日小结**

| 函数 | 作用 | 关键参数 |
| --- | --- | --- |
| **pd.read_csv** | 读 CSV | encoding/sep/header/usecols/na_values |
| **pd.read_json** | 读 JSON | orient/force_ascii |
| **pd.read_excel** | 读 Excel | sheet_name(需 openpyxl) |
| **pd.read_sql** | 读 SQL 查询 | sql/con(连接对象) |
| **requests.get** | 拉 API 数据 | url/status_code/response.json() |
| **df.to_csv** | 写 CSV | index=False/encoding |
| **df.to_json** | 写 JSON | orient/force_ascii |
| **df.to_sql** | 写数据库 | if_exists(replace/append) |

**三个小练习**

1. 用 `io.StringIO` 构造一份含缺失值 "N/A" 的 CSV,读取时把它转成 `NaN`,再打印 `df.isna()`。
2. 用 `to_json(orient="columns", force_ascii=False)` 导出数据,对比与 `orient="records"` 的差异。
3. 建一个内存 SQLite 库,通过 `to_sql` 写入两张表,再用 `read_sql` 做 JOIN 查询并打印结果。

**常见报错**

- **UnicodeDecodeError**: CSV 编码不是 utf-8,尝试 `encoding="gbk"`。
- **连接未关闭导致锁表**: 读/写完数据库后务必 `conn.close()` 或改用 `with` 上下文。
- **API 请求未检查 status_code**: 若返回 404/500,直接 `.json()` 会报错,应先判 status_code。
- **to_csv 后多出一列序号**: 忘加 `index=False`。
